# EmotNet Training Pipeline

This notebook covers the training of **EmotNet**, which processes user speech transcripts and self-talk inputs to detect **Social Anxiety**, **GAD**, and **Childhood Depression**.

## Modality & Scope
- Input: Token IDs and attention masks representing speech transcripts (sequence length = 64)
- Output: Multi-class probabilities (0 = Typical/Control, 1 = Worry, 2 = Perfectionism, 3 = Sadness)
- Base Network: **Lightweight Embedding Dense MLP** (optimized for fast CPU/GPU convergence and Android latency)

## Target Runtime
- Google Colab (Free T4 GPU)

## Step 1: Environment Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas numpy matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Dataset Generation & Synthesizing
We generate synthetic transcripts mapping semantic categories based on key indicators: Control (0), Worry/Anxiety (1), Perfectionism (2), Sadness (3).

In [ ]:
def generate_synthetic_transcripts(num_samples=1000):
    np.random.seed(42)
    X_ids = np.random.randint(100, 20000, (num_samples, 64)).astype(np.int32)
    X_mask = np.ones((num_samples, 64), dtype=np.int32)
    y = np.array([i % 4 for i in range(num_samples)], dtype=np.int32)
    
    # Inject semantic tokens representing clinical domains
    # 1000 = worry/anxious/scared, 2000 = must/perfect/error, 3000 = sad/unhappy/tired
    for i in range(num_samples):
        label = i % 4
        if label == 1: # Worry
            X_ids[i, 0:10] = 1000
        elif label == 2: # Perfectionism
            X_ids[i, 0:10] = 2000
        elif label == 3: # Sadness
            X_ids[i, 0:10] = 3000
            
    return X_ids, X_mask, y

X_ids_train, X_mask_train, y_train = generate_synthetic_transcripts(1000)
X_ids_val, X_mask_val, y_val = generate_synthetic_transcripts(200)
print("Synthetic transcripts generated:", X_ids_train.shape, y_train.shape)

## Step 3: Model Architecture Design
We define a lightweight dual-input Keras embedding classifier. It takes `input_ids` and `attention_mask` as separate parameters, matching the exact signature expected by the Android codebase.

In [ ]:
input_ids = layers.Input(shape=(64,), dtype=tf.int32, name="input_ids")
attention_mask = layers.Input(shape=(64,), dtype=tf.int32, name="attention_mask")

embedding_layer = layers.Embedding(input_dim=30000, output_dim=16)
emb = embedding_layer(input_ids)

# Combine inputs and pool
x = layers.GlobalAveragePooling1D()(emb)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(4, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(model.summary())

## Step 4: Model Training

In [ ]:
history = model.fit(
    [X_ids_train, X_mask_train], y_train,
    validation_data=([X_ids_val, X_mask_val], y_val),
    epochs=20,
    batch_size=32
)

## Step 5: Export to TFLite (Float16 Quantized)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

output_path = 'output/seren_emotnet.tflite'
os.makedirs('output', exist_ok=True)
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model successfully exported to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

## Step 6: Automated Behavioral Validation Check

In [ ]:
print("\n--- Running Behavioral Validation Check ---")
interpreter = tf.lite.Interpreter(model_path=output_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

inp1_ids = np.zeros((1, 64), dtype=np.int32)
inp1_ids[0, 0:10] = 1000 # worry tokens

inp2_ids = np.zeros((1, 64), dtype=np.int32)
inp2_ids[0, 0:10] = 2000 # perfectionism tokens

inp3_ids = np.zeros((1, 64), dtype=np.int32)
inp3_ids[0, 0:10] = 3000 # sadness tokens

mask = np.ones((1, 64), dtype=np.int32)

# Map inputs to the exact TFLite indices matching input names
test_inputs = [inp1_ids, inp2_ids, inp3_ids]
outputs = []
for ids in test_inputs:
    interpreter.set_tensor(input_details[0]['index'], ids)
    interpreter.set_tensor(input_details[1]['index'], mask)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    outputs.append(out.flatten())
    print(f"Input -> Output probabilities: {out.flatten()}")

outputs = np.array(outputs)
std_devs = np.std(outputs, axis=0)
max_std = np.max(std_devs)
print(f"Max standard deviation across outputs: {max_std:.4f}")
assert max_std > 0.05, "FAILED: EmotNet outputs are identical! The weights did not train or converge."
print("PASSED: EmotNet behavioral check succeeded.")